# Three-threshold causal live development

This notebook turns the linked GBPUSD.r hierarchy into a causal, stateful read.

The immutable lineage key is the initiating small DCC. If medium confirms the same target direction, the exact medium event becomes the Stage-2 hand-off without replacing the original seed. Later small DCCs are nested activity inside the active higher-scale contest.

All live features are stamped at the observation bar. Propagated/rejected and terminal depth are attached later for diagnostic comparison and remain hindsight until their resolution timestamps.

In [1]:
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score

from dc_basics.core import dc_live_features, parse_dc_events
from dc_basics.data import resolve_data_root

DATA_ROOT = resolve_data_root()
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
SMALL_THETA = 0.002
MEDIUM_THETA = 0.004
LARGE_THETA = 0.006
CHECKPOINT_BARS = [0, 5, 10, 20, 40, 80, 160, 320, 640]
# An AUC is reported only when both later outcomes have at least this many survivors.
MIN_AUC_CLASS_COUNT = 20
# Checkpoints at which live geometry is compared with completed initiation features.
INCREMENTAL_CHECKPOINTS = [0, 10, 20, 40, 80]
CASEBOOK_BARS_BEFORE = 120
CASEBOOK_BARS_AFTER = 120

if not 0.0 < SMALL_THETA < MEDIUM_THETA < LARGE_THETA:
    raise ValueError("Thresholds must satisfy 0 < small < medium < large.")

print(f"Symbol: {SYMBOL}")
print(f"Thresholds: {SMALL_THETA:.2%}, {MEDIUM_THETA:.2%}, {LARGE_THETA:.2%}")

Symbol: GBPUSD.r
Thresholds: 0.20%, 0.40%, 0.60%


In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"


from dc_basics.enums import DcMode


from dc_basics.enums import AttemptOutcome


class TerminalOutcome(StrEnum):
    STOPPED_BELOW_MEDIUM = "stopped_below_medium"
    STOPPED_BELOW_LARGE = "stopped_below_large"
    REACHED_LARGE = "reached_large"
    AMBIGUOUS_SMALL_MEDIUM = "ambiguous_small_medium"
    AMBIGUOUS_MEDIUM_LARGE = "ambiguous_medium_large"
    UNRESOLVED_BELOW_MEDIUM = "unresolved_below_medium"
    UNRESOLVED_BELOW_LARGE = "unresolved_below_large"
    UNLINKED_AFTER_MEDIUM = "unlinked_after_medium"


class Stage(StrEnum):
    SMALL_TO_MEDIUM = "small_to_medium"
    MEDIUM_TO_LARGE = "medium_to_large"


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE)
SCALE_SETTINGS = {
    Scale.SMALL: {"prefix": "S", "theta": SMALL_THETA},
    Scale.MEDIUM: {"prefix": "M", "theta": MEDIUM_THETA},
    Scale.LARGE: {"prefix": "L", "theta": LARGE_THETA},
}
STAGE_SETTINGS = {
    Stage.SMALL_TO_MEDIUM: {"lower": Scale.SMALL, "upper": Scale.MEDIUM, "attained_depth": 0},
    Stage.MEDIUM_TO_LARGE: {"lower": Scale.MEDIUM, "upper": Scale.LARGE, "attained_depth": 1},
}
STAGE_ORDER = (Stage.SMALL_TO_MEDIUM, Stage.MEDIUM_TO_LARGE)
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

## Stateful information contract

The read moves through three states:

    idle
      -> Stage 1 after a small DCC leaves broad agreement
      -> Stage 2 only if the exact medium DCC confirms the target
      -> terminal when the active lower scale rejects or the upper scale propagates

The small seed event ID and seed timestamp never change. At Stage 2, the medium event ID is a hand-off key. A small DCC occurring after that hand-off updates nested activity but never re-roots the lineage.

The resolution bar is excluded from every live trajectory. Consequently, the stage outcome is still unknown on every feature row.

## Load prices and reconstruct the three independent DC clocks

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no 'close' column.")
prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Bars: 258,573
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


In [4]:
from dc_basics.events import prepare_event_ledger


event_ledgers = {}
live_frames = {}
mode_frame = pd.DataFrame(index=prices.index)
for scale in SCALE_ORDER:
    settings = SCALE_SETTINGS[scale]
    raw_events = parse_dc_events(prices=prices, theta=settings["theta"])
    ledger = prepare_event_ledger(raw_events, settings["prefix"], scale, settings["theta"])
    live = dc_live_features(prices, theta=settings["theta"])
    event_ledgers[scale] = ledger
    live_frames[scale] = live
    mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
    mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(ledger.set_index("dcc_idx")["event_id"])

clock_summary = pd.DataFrame.from_records([
    {
        "scale": scale.value,
        "threshold": SCALE_SETTINGS[scale]["theta"],
        "events": len(event_ledgers[scale]),
        "first_mode_time": live_frames[scale]["direction"].first_valid_index(),
    }
    for scale in SCALE_ORDER
]).set_index("scale")
clock_summary.style.format({"threshold": "{:.2%}", "events": "{:,}"})

,threshold,events,first_mode_time
scale,,,
small,0.20%,"2,697",2023-05-22 16:15:00+02:00
medium,0.40%,839,2023-05-22 17:50:00+02:00
large,0.60%,392,2023-05-23 11:30:00+02:00


## Reconstruct adjacent contests and pin broad-agreement seeds

An adjacent contest starts when the lower-scale DCC moves against the upper mode. It resolves when the lower mode returns first or the upper mode confirms first. Broad-agreement ancestry is imposed only when selecting the original small seed.

In [5]:
from dc_basics.hierarchy import build_adjacent_attempts


In [6]:
small_medium_attempts = build_adjacent_attempts(mode_frame, Scale.SMALL, Scale.MEDIUM)
medium_large_attempts = build_adjacent_attempts(mode_frame, Scale.MEDIUM, Scale.LARGE)

previous_modes = mode_frame[["small_mode", "medium_mode", "large_mode"]].shift(1)
seed_attempts = small_medium_attempts.copy()
seed_times = pd.DatetimeIndex(seed_attempts["start_time"])
for scale in SCALE_ORDER:
    seed_attempts[f"prior_{scale.value}_mode"] = previous_modes.loc[seed_times, f"{scale.value}_mode"].to_numpy()
seed_attempts["broad_agreement_before_seed"] = (
    seed_attempts["prior_small_mode"].eq(seed_attempts["prior_medium_mode"])
    & seed_attempts["prior_medium_mode"].eq(seed_attempts["prior_large_mode"])
)
broad_seeds = seed_attempts.loc[seed_attempts["broad_agreement_before_seed"]].copy()

print(f"All small-to-medium attempts: {len(seed_attempts):,}")
print(f"Broad-agreement small seeds: {len(broad_seeds):,}")

All small-to-medium attempts: 1,768
Broad-agreement small seeds: 1,279


## Link the two stages by exact event genealogy

In [7]:
from dc_basics.lineage import build_three_scale_linkages


linkages = build_three_scale_linkages(broad_seeds, medium_large_attempts)
resolved_linkages = linkages.loc[linkages["terminal_depth"].notna()].copy()
stage_one_resolution_position = resolved_linkages["terminal_position"].where(
    resolved_linkages["terminal_depth"].eq(0), resolved_linkages["medium_position"]
)
stage_one_has_live_bar = pd.to_numeric(stage_one_resolution_position).gt(resolved_linkages["seed_position"])
stage_two_has_live_bar = (
    resolved_linkages["terminal_depth"].eq(0)
    | pd.to_numeric(resolved_linkages["terminal_position"]).gt(
        pd.to_numeric(resolved_linkages["medium_position"])
    )
)
strict_clean_linkages = resolved_linkages.loc[
    resolved_linkages["terminal_label_hindsight_at_seed"]
    & ~resolved_linkages["small_medium_same_bar"]
    & ~resolved_linkages["medium_large_same_bar"]
    & stage_one_has_live_bar
    & stage_two_has_live_bar
].copy()
if strict_clean_linkages.empty:
    raise ValueError("No strict-clean linked lineages were found.")
if not stage_one_has_live_bar.loc[strict_clean_linkages.index].all():
    raise AssertionError("A strict-clean Stage-1 lineage has no pre-resolution bar.")
strict_stage_two = strict_clean_linkages["terminal_depth"].ge(1)
strict_stage_two_positions = strict_clean_linkages.loc[strict_stage_two]
if not strict_stage_two_positions["terminal_position"].gt(strict_stage_two_positions["medium_position"]).all():
    raise AssertionError("A strict-clean Stage-2 lineage has no pre-resolution bar.")

linkage_audit = pd.Series({
    "broad_agreement_seeds": len(linkages),
    "resolved_lineages": len(resolved_linkages),
    "same_bar_small_medium": int(resolved_linkages["small_medium_same_bar"].sum()),
    "same_bar_medium_large": int(resolved_linkages["medium_large_same_bar"].sum()),
    "strict_clean_lineages": len(strict_clean_linkages),
})
linkage_audit

broad_agreement_seeds    1279
resolved_lineages        1278
same_bar_small_medium      36
same_bar_medium_large      15
strict_clean_lineages    1235
dtype: int64

## Construct causal geometry before the resolution bars

For each scale, the running extreme is updated only from prices already observed in the current DC mode. Progress toward the next DCC is the signed reversal from that causal extreme divided by the upper threshold.

Progress is not expected to start at zero. A Stage-1 seed has already travelled far enough to confirm the small threshold. A Stage-2 hand-off has already travelled far enough to confirm the medium threshold.

In [8]:
from dc_basics.live import build_causal_running_extreme


running_extremes = {
    scale: build_causal_running_extreme(prices, live_frames[scale])
    for scale in SCALE_ORDER
}
small_dcc_flag = (mode_frame["small_upward_dcc"] | mode_frame["small_downward_dcc"]).astype(np.int64)
small_dcc_cumulative = small_dcc_flag.cumsum().to_numpy(dtype=np.int64)
configuration_series = mode_frame[["small_mode", "medium_mode", "large_mode"]].astype(str).agg("/".join, axis=1)

if any(extreme.dropna().empty for extreme in running_extremes.values()):
    raise ValueError("A causal running-extreme series is empty.")

In [9]:
from dc_basics.audit import audit_live_panel
from dc_basics.lineage import build_wide_stage_table
from dc_basics.live_panel import LivePanelContext, build_live_panel


mode_frame["configuration"] = configuration_series
dcc_cumulative = {Scale.SMALL: small_dcc_cumulative}
for scale in (Scale.MEDIUM,):
    dcc_flag = (
        mode_frame[f"{scale.value}_upward_dcc"]
        | mode_frame[f"{scale.value}_downward_dcc"]
    ).astype(np.int64)
    dcc_cumulative[scale] = dcc_flag.cumsum().to_numpy(dtype=np.int64)
live_context = LivePanelContext(
    prices=prices,
    mode_frame=mode_frame,
    live_frames=live_frames,
    running_extremes=running_extremes,
    thresholds={scale: SCALE_SETTINGS[scale]["theta"] for scale in SCALE_ORDER},
    dcc_cumulative=dcc_cumulative,
    scale_order=SCALE_ORDER,
    hierarchy_columns=("configuration",),
    event_ledgers=event_ledgers,
)
live_stages = build_wide_stage_table(strict_clean_linkages, SCALE_ORDER)


In [10]:
expected_stage_counts = {
    stage.value: int(strict_clean_linkages["terminal_depth"].ge(stage_index).sum())
    for stage_index, stage in enumerate(STAGE_ORDER)
}
live_panel = build_live_panel(live_stages, strict_clean_linkages, live_context, STAGE_SETTINGS)
live_panel_audit = audit_live_panel(
    live_panel,
    expected_stage_counts=expected_stage_counts,
    require_seed_identity=True,
)
live_panel_audit.to_frame()


,value
lineages,1235.00000
active_pre_resolution_rows,145235.00000
maximum_pre_resolution_progress,0.99999
seed_ids_that_changed,0.00000
medium_to_large_lineages,572.00000
small_to_medium_lineages,1235.00000
maximum_nested_small_dccs,9.00000
maximum_nested_medium_dccs,0.00000


## Information-timing audit

The stage outcome timestamp answers the adjacent question currently being contested. The terminal timestamp answers how far the complete lineage eventually travelled. They can differ: at a medium hand-off, Stage 1 is known to have propagated while the final depth is still unknown.

In [11]:
information_contract = pd.DataFrame([
    ("lineage_id", True, "immutable initiating small event ID"),
    ("stage_start_event_id", True, "small seed in Stage 1; exact medium hand-off in Stage 2"),
    ("attained_depth", True, "largest scale already confirmed at the observation bar"),
    ("elapsed_bars", True, "bars since the current stage began"),
    ("lineage_age_bars", True, "bars since the immutable small seed"),
    ("seed_leg_tmv / seed_leg_t / seed_leg_r", True, "completed small leg confirmed by the seed DCC"),
    ("stage_start_leg_tmv / _t / _r", True, "completed leg confirmed by the stage-start DCC (seed or medium hand-off)"),
    ("lower_tmv_live / lower_t_live / lower_r_live", True, "live lower-scale event geometry"),
    ("upper_tmv_live / upper_t_live / upper_r_live", True, "live upper-scale event geometry"),
    ("target_progress", True, "current fraction of the upper DCC threshold traversed"),
    ("max_target_progress", True, "largest causal progress observed in the active stage"),
    ("nested_small_dccs", True, "small DCCs strictly after the stage start; zero by construction in Stage 1"),
    ("sub_lower_dccs / sub_lower_dcc_rate", True, "DCCs on clocks below the active lower clock, and per bar"),
    ("future_stage_outcome", False, "propagated or rejected at the stage resolution DCC"),
    ("future_terminal_depth", False, "maximum confirmed scale, known only at terminal resolution"),
], columns=["column", "known_on_feature_bar", "economic_meaning"])
information_contract

,column,known_on_feature_bar,economic_meaning
0,lineage_id,True,immutable initiating small event ID
1,stage_start_event_id,True,small seed in Stage 1; exact medium hand-off i...
2,attained_depth,True,largest scale already confirmed at the observa...
3,elapsed_bars,True,bars since the current stage began
4,lineage_age_bars,True,bars since the immutable small seed
5,seed_leg_tmv / seed_leg_t / seed_leg_r,True,completed small leg confirmed by the seed DCC
6,stage_start_leg_tmv / _t / _r,True,completed leg confirmed by the stage-start DCC...
7,lower_tmv_live / lower_t_live / lower_r_live,True,live lower-scale event geometry
8,upper_tmv_live / upper_t_live / upper_r_live,True,live upper-scale event geometry
9,target_progress,True,current fraction of the upper DCC threshold tr...


## Live Audit 1: fixed checkpoints and survival conditioning

At checkpoint k, only stages that remain unresolved after k bars appear. Later checkpoints are selected survivor cohorts. Their propagation rates must not be read as forecasts made at the original seed.

In [12]:
checkpoint_rows = live_panel.loc[live_panel["elapsed_bars"].isin(CHECKPOINT_BARS)].copy()
stage_attempt_counts = live_panel.groupby("stage")["lineage_id"].nunique()

checkpoint_survival = checkpoint_rows.groupby(["stage", "elapsed_bars"], sort=True).agg(
    surviving_attempts=("lineage_id", "nunique"),
    later_propagation_rate=(
        "future_stage_outcome",
        lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean(),
    ),
    lower_tmv_median=("lower_tmv_live", "median"),
    lower_t_median=("lower_t_live", "median"),
    lower_r_median=("lower_r_live", "median"),
    target_progress_median=("target_progress", "median"),
    max_target_progress_median=("max_target_progress", "median"),
    max_favorable_excursion_median=("max_favorable_excursion_theta", "median"),
    max_adverse_excursion_median=("max_adverse_excursion_theta", "median"),
    sub_lower_dccs_median=("sub_lower_dccs", "median"),
    lineage_age_median=("lineage_age_bars", "median"),
).reset_index()
checkpoint_survival["survivor_share"] = checkpoint_survival.apply(
    lambda row: row["surviving_attempts"] / stage_attempt_counts[row["stage"]], axis=1
)
checkpoint_survival.set_index(["stage", "elapsed_bars"]).style.format({
    "surviving_attempts": "{:,}",
    "later_propagation_rate": "{:.2%}",
    "lower_tmv_median": "{:.3f}",
    "lower_t_median": "{:.1f}",
    "lower_r_median": "{:.3e}",
    "target_progress_median": "{:.2%}",
    "max_target_progress_median": "{:.2%}",
    "max_favorable_excursion_median": "{:.3f}",
    "max_adverse_excursion_median": "{:.3f}",
    "sub_lower_dccs_median": "{:.1f}",
    "lineage_age_median": "{:.0f}",
    "survivor_share": "{:.2%}",
})

In [13]:
checkpoint_outcome_summary = checkpoint_rows.groupby(
    ["stage", "elapsed_bars", "future_stage_outcome", "target_mode"], sort=True
).agg(
    surviving_attempts=("lineage_id", "size"),
    lower_tmv_median=("lower_tmv_live", "median"),
    lower_t_median=("lower_t_live", "median"),
    lower_r_median=("lower_r_live", "median"),
    target_progress_median=("target_progress", "median"),
    max_target_progress_median=("max_target_progress", "median"),
    max_favorable_excursion_median=("max_favorable_excursion_theta", "median"),
    max_adverse_excursion_median=("max_adverse_excursion_theta", "median"),
    sub_lower_dccs_median=("sub_lower_dccs", "median"),
    lineage_age_median=("lineage_age_bars", "median"),
)
checkpoint_outcome_summary.style.format({
    "surviving_attempts": "{:,}",
    "lower_tmv_median": "{:.3f}",
    "lower_t_median": "{:.1f}",
    "lower_r_median": "{:.3e}",
    "target_progress_median": "{:.2%}",
    "max_target_progress_median": "{:.2%}",
    "max_favorable_excursion_median": "{:.3f}",
    "max_adverse_excursion_median": "{:.3f}",
    "sub_lower_dccs_median": "{:.1f}",
    "lineage_age_median": "{:.0f}",
})

## Live Audit 2: outcome overlap at each checkpoint

Orientation-free AUC measures univariate ordering between the two later stage outcomes among checkpoint survivors. The outcome label is hindsight, the feature value is causal, and the calculation is in-sample diagnostic. A rising AUC can reflect developing information, survival selection, or both.

Each table shows the propagated and rejected survivor counts beside the AUCs. An AUC is reported only when both groups have at least `MIN_AUC_CLASS_COUNT` survivors; otherwise the cell says `insufficient_sample`. A cell says `constant_by_construction` when the feature cannot vary inside a pre-resolution panel: the stage's own lower and upper clocks cannot fire before the resolution bar, so Stage 1 has no nested small DCCs.

In [14]:
from dc_basics.live_panel import (
    auc_table_with_counts,
    constant_by_construction_features,
    summarize_checkpoint_overlap,
)

LIVE_FEATURE_COLUMNS = [
    "lower_tmv_live",
    "lower_t_live",
    "lower_r_live",
    "upper_tmv_live",
    "upper_t_live",
    "upper_r_live",
    "target_progress",
    "max_target_progress",
    "max_favorable_excursion_theta",
    "max_adverse_excursion_theta",
    "sub_lower_dccs",
    "sub_lower_dcc_rate",
    "lineage_age_bars",
]
CONSTANT_FEATURES = constant_by_construction_features(STAGE_SETTINGS, SCALE_ORDER)

live_overlap_summary = summarize_checkpoint_overlap(
    checkpoint_rows,
    LIVE_FEATURE_COLUMNS,
    min_class_count=MIN_AUC_CLASS_COUNT,
    constant_by_construction=CONSTANT_FEATURES,
)
for stage in Stage:
    print(stage.value)
    display(auc_table_with_counts(
        live_overlap_summary.loc[live_overlap_summary["stage"].eq(stage.value)],
        features=LIVE_FEATURE_COLUMNS,
    ))

small_to_medium


,propagated_survivors,rejected_survivors,lower_tmv_live,lower_t_live,lower_r_live,upper_tmv_live,upper_t_live,upper_r_live,target_progress,max_target_progress,max_favorable_excursion_theta,max_adverse_excursion_theta,sub_lower_dccs,sub_lower_dcc_rate,lineage_age_bars
elapsed_bars,,,,,,,,,,,,,,,
0,572,663,0.556,0.520,0.526,0.517,0.500,0.510,0.571,0.571,constant_in_sample,constant_in_sample,constant_by_construction,constant_by_construction,constant_in_sample
5,507,641,0.650,0.514,0.575,0.542,0.506,0.529,0.654,0.605,0.598,0.586,constant_by_construction,constant_by_construction,constant_in_sample
10,436,571,0.686,0.509,0.604,0.549,0.509,0.532,0.685,0.619,0.614,0.599,constant_by_construction,constant_by_construction,constant_in_sample
20,361,466,0.731,0.525,0.662,0.558,0.516,0.535,0.733,0.664,0.662,0.640,constant_by_construction,constant_by_construction,constant_in_sample
40,235,326,0.712,0.509,0.653,0.566,0.534,0.513,0.712,0.657,0.655,0.636,constant_by_construction,constant_by_construction,constant_in_sample
80,131,173,0.735,0.523,0.674,0.552,0.519,0.519,0.734,0.699,0.700,0.623,constant_by_construction,constant_by_construction,constant_in_sample
160,44,67,0.747,0.520,0.723,0.588,0.514,0.554,0.737,0.730,0.739,0.593,constant_by_construction,constant_by_construction,constant_in_sample
320,1,2,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,constant_by_construction,constant_by_construction,insufficient_sample


medium_to_large


,propagated_survivors,rejected_survivors,lower_tmv_live,lower_t_live,lower_r_live,upper_tmv_live,upper_t_live,upper_r_live,target_progress,max_target_progress,max_favorable_excursion_theta,max_adverse_excursion_theta,sub_lower_dccs,sub_lower_dcc_rate,lineage_age_bars
elapsed_bars,,,,,,,,,,,,,,,
0,356,216,0.520,0.531,0.533,0.505,0.526,0.520,0.531,0.531,constant_in_sample,constant_in_sample,constant_in_sample,constant_in_sample,0.509
5,325,214,0.630,0.525,0.552,0.525,0.526,0.542,0.634,0.572,0.587,0.601,0.513,0.513,0.517
10,300,213,0.674,0.518,0.561,0.541,0.520,0.561,0.674,0.591,0.614,0.632,0.535,0.535,0.526
20,259,200,0.712,0.511,0.594,0.548,0.536,0.584,0.713,0.605,0.625,0.676,0.598,0.598,0.534
40,207,167,0.722,0.515,0.624,0.565,0.504,0.568,0.718,0.619,0.643,0.655,0.610,0.610,0.537
80,142,129,0.770,0.523,0.690,0.592,0.513,0.588,0.767,0.673,0.686,0.673,0.625,0.625,0.506
160,81,91,0.767,0.543,0.713,0.599,0.549,0.558,0.763,0.694,0.713,0.696,0.546,0.546,0.513
320,20,27,0.733,0.571,0.626,0.517,0.522,0.531,0.730,0.630,0.667,0.613,0.501,0.501,0.514
640,2,2,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample,insufficient_sample


## Live Audit 2b: live geometry versus completed initiation features

Does the live path add anything beyond what was already known when the stage began? Two feature sets are compared on the **same survivor cohort** at each checkpoint:

- **completed initiation features:** TMV, T and R of the completed small leg confirmed by the seed DCC, plus, in Stage 2, the completed medium leg confirmed by the hand-off DCC. Each is stamped with its own information time, which equals the seed or hand-off time;
- **live geometry:** progress, excursions and live TMV/R on the observation bar, plus nested small activity in Stage 2.

`best_*_auc` is the best univariate in-sample AUC of each set. The `oos_auc_*` columns come from three logistic models (initiation only, initiation plus live, live only) scored with forward-chaining folds ordered by stage start, so each attempt is scored by a model fitted only on earlier attempts. `oos_auc_gain` is the plus-live AUC minus the initiation-only AUC. Initiation-only AUCs often fall below 0.5, i.e. worse than chance, which inflates that difference. `oos_auc_gain_over_floor` therefore measures the plus-live AUC against the better of 0.5 and the initiation-only model, and `oos_auc_live_only` shows what the live features achieve on their own. Credit the live path with information only when the floor gain is positive. At elapsed 0 there is no live path yet, so any "gain" there reflects the state at stage start, not path development. This answers chapter 13 exercise 5. It is still a diagnostic of incremental information, not a forecast, and survivor cohorts shrink at later checkpoints.

In [15]:
from dc_basics.live_panel import compare_initiation_and_live_features

INITIATION_FEATURES = {
    Stage.SMALL_TO_MEDIUM.value: ["seed_leg_tmv", "seed_leg_t", "seed_leg_r"],
    Stage.MEDIUM_TO_LARGE.value: [
        "seed_leg_tmv", "seed_leg_t", "seed_leg_r",
        "stage_start_leg_tmv", "stage_start_leg_t", "stage_start_leg_r",
    ],
}
LIVE_COMPARISON_FEATURES = [
    "target_progress", "max_target_progress", "lower_tmv_live", "lower_r_live",
    "max_favorable_excursion_theta", "max_adverse_excursion_theta",
]

incremental_records = []
for stage in STAGE_ORDER:
    live_features = LIVE_COMPARISON_FEATURES + (
        ["sub_lower_dccs"] if stage != Stage.SMALL_TO_MEDIUM else []
    )
    for elapsed_bars in INCREMENTAL_CHECKPOINTS:
        cohort = checkpoint_rows.loc[
            checkpoint_rows["stage"].eq(stage.value) & checkpoint_rows["elapsed_bars"].eq(elapsed_bars)
        ]
        incremental_records.append({
            "stage": stage.value,
            "elapsed_bars": elapsed_bars,
            **compare_initiation_and_live_features(
                cohort,
                INITIATION_FEATURES[stage.value],
                live_features,
                min_class_count=MIN_AUC_CLASS_COUNT,
            ),
        })

incremental_summary = pd.DataFrame.from_records(incremental_records).set_index(["stage", "elapsed_bars"])
incremental_summary.style.format({
    "best_initiation_auc": "{:.3f}",
    "best_live_auc": "{:.3f}",
    "oos_auc_initiation": "{:.3f}",
    "oos_auc_initiation_plus_live": "{:.3f}",
    "oos_auc_gain": "{:+.3f}",
    "oos_auc_live_only": "{:.3f}",
    "oos_auc_gain_over_floor": "{:+.3f}",
}, na_rep="—")

## Live Audit 3: survival-conditioned paths

Color denotes the later stage outcome and line style denotes the target direction. Every plotted y-value was available at its elapsed bar; the grouping color remains hindsight.

In [16]:
def plot_live_stage_trajectories(panel: pd.DataFrame, stage: Stage, max_elapsed_bars: int) -> go.Figure:
    # Stage 1's lower clock is small: nothing fires below it, so a nested-count panel would be flat.
    fourth_feature = (
        {"max_adverse_excursion_theta": "Max adverse excursion (lower-theta units)"}
        if stage == Stage.SMALL_TO_MEDIUM
        else {"sub_lower_dccs": "Small DCCs since the medium hand-off"}
    )
    features = {
        "target_progress": "Current progress toward upper DCC",
        "max_target_progress": "Maximum progress toward upper DCC",
        "lower_r_live": "Live lower-scale R",
        **fourth_feature,
    }
    selected = panel.loc[panel["stage"].eq(stage.value) & panel["elapsed_bars"].le(max_elapsed_bars)].copy()
    summary = selected.groupby(["elapsed_bars", "future_stage_outcome", "target_mode"], sort=True).agg(
        survivor_count=("lineage_id", "size"),
        **{f"{feature}_median": (feature, "median") for feature in features},
    ).reset_index()
    figure = make_subplots(rows=2, cols=2, subplot_titles=list(features.values()))
    colors = {AttemptOutcome.PROPAGATED.value: "#2563eb", AttemptOutcome.REJECTED.value: "#f97316"}
    dashes = {DcMode.UP.value: "solid", DcMode.DOWN.value: "dash"}

    for feature_number, (feature, label) in enumerate(features.items()):
        row_number = feature_number // 2 + 1
        column_number = feature_number % 2 + 1
        for (outcome, target_mode), group in summary.groupby(["future_stage_outcome", "target_mode"], sort=True):
            figure.add_trace(go.Scattergl(
                x=group["elapsed_bars"],
                y=group[f"{feature}_median"],
                mode="lines",
                name=f"{outcome} | target {target_mode}",
                legendgroup=f"{outcome}_{target_mode}",
                showlegend=feature_number == 0,
                line={"color": colors[outcome], "dash": dashes[target_mode], "width": 2},
                customdata=group[["survivor_count"]].to_numpy(),
                hovertemplate=(
                    "elapsed=%{x:.0f}<br>median=%{y:.4g}<br>"
                    "survivors=%{customdata[0]:,.0f}<extra></extra>"
                ),
            ), row=row_number, col=column_number)
        figure.update_xaxes(title_text="Elapsed bars", row=row_number, col=column_number)
        figure.update_yaxes(title_text=label, row=row_number, col=column_number)
        if feature in {"target_progress", "max_target_progress"}:
            figure.add_hline(y=1.0, line={"color": "#64748b", "dash": "dot"}, row=row_number, col=column_number)

    figure.update_layout(
        title=f"{SYMBOL}: {stage.value} live paths — color is hindsight outcome",
        template="plotly_white",
        height=820,
        hovermode="closest",
        legend={"orientation": "h", "y": 1.08, "x": 0.0},
    )
    return figure


plot_live_stage_trajectories(live_panel, Stage.SMALL_TO_MEDIUM, 200).show()
plot_live_stage_trajectories(live_panel, Stage.MEDIUM_TO_LARGE, 400).show()

## Live Audit 4: pinned-seed and hand-off casebook

One recent propagated and rejected example is selected for each stage. Selection and color are hindsight. Within each plot, the thick segment contains only unresolved feature bars; the resolution marker is the first bar where the stage outcome becomes known.

In [17]:
stage_attempts = live_panel.groupby(["lineage_id", "stage"], sort=False).agg(
    seed_time=("seed_time", "first"),
    stage_start_event_id=("stage_start_event_id", "first"),
    stage_start_time=("stage_start_time", "first"),
    start_position=("bar_position", "min"),
    target_mode=("target_mode", "first"),
    future_stage_outcome=("future_stage_outcome", "first"),
    stage_outcome_information_time=("stage_outcome_information_time", "first"),
    future_terminal_depth=("future_terminal_depth", "first"),
    terminal_outcome_information_time=("terminal_outcome_information_time", "first"),
    pre_resolution_rows=("bar_position", "size"),
).reset_index()
stage_attempts["resolution_position"] = stage_attempts["start_position"] + stage_attempts["pre_resolution_rows"]
stage_attempts["resolution_bars"] = stage_attempts["pre_resolution_rows"]

sorted_stage_attempts = stage_attempts.sort_values("stage_start_time")
casebook_attempts = sorted_stage_attempts.groupby(
    ["stage", "future_stage_outcome"], sort=True, group_keys=False
).tail(1)
casebook_attempts = casebook_attempts.sort_values(["stage", "future_stage_outcome"])
casebook_attempts[[
    "lineage_id", "stage", "stage_start_event_id", "stage_start_time", "target_mode",
    "future_stage_outcome", "stage_outcome_information_time", "future_terminal_depth",
    "resolution_bars",
]]

,lineage_id,stage,stage_start_event_id,stage_start_time,target_mode,future_stage_outcome,stage_outcome_information_time,future_terminal_depth,resolution_bars
1805,S2694,medium_to_large,M838,2026-11-05 23:05:00+02:00,down,propagated,2026-11-06 08:45:00+02:00,2,116
1799,S2684,medium_to_large,M836,2026-10-30 17:00:00+02:00,down,rejected,2026-11-02 10:35:00+02:00,1,211
1804,S2694,small_to_medium,S2694,2026-11-05 21:15:00+02:00,down,propagated,2026-11-05 23:05:00+02:00,2,22
1806,S2695,small_to_medium,S2695,2026-11-06 17:20:00+02:00,up,rejected,2026-11-06 21:45:00+02:00,0,53


In [18]:
def plot_live_stage_casebook(close: pd.Series, panel: pd.DataFrame, attempts: pd.DataFrame) -> go.Figure:
    figure = make_subplots(
        rows=len(attempts),
        cols=1,
        vertical_spacing=min(0.06, 0.20 / len(attempts)),
        subplot_titles=[
            (
                f"{attempt.stage} | {attempt.future_stage_outcome} | "
                f"seed={attempt.lineage_id} | stage event={attempt.stage_start_event_id}"
            )
            for attempt in attempts.itertuples(index=False)
        ],
    )
    outcome_colors = {
        AttemptOutcome.PROPAGATED.value: "#2563eb",
        AttemptOutcome.REJECTED.value: "#f97316",
    }

    for row_number, attempt in enumerate(attempts.itertuples(index=False), start=1):
        start_position = int(attempt.start_position)
        resolution_position = int(attempt.resolution_position)
        window_start = max(0, start_position - CASEBOOK_BARS_BEFORE)
        window_end = min(len(close), resolution_position + CASEBOOK_BARS_AFTER + 1)
        price_window = close.iloc[window_start:window_end]
        active_path = panel.loc[
            panel["lineage_id"].eq(attempt.lineage_id)
            & panel["stage"].eq(attempt.stage)
        ]

        figure.add_trace(go.Scattergl(
            x=price_window.index,
            y=price_window.to_numpy(),
            mode="lines",
            name="Close",
            legendgroup="close",
            showlegend=row_number == 1,
            line={"color": "#94a3b8", "width": 1.0},
            hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)
        figure.add_trace(go.Scattergl(
            x=active_path["time"],
            y=active_path["price"],
            mode="lines",
            name="Active unresolved stage",
            legendgroup="active",
            showlegend=row_number == 1,
            line={"color": outcome_colors[attempt.future_stage_outcome], "width": 3.0},
            customdata=active_path[[
                "elapsed_bars", "target_progress", "max_target_progress",
                "lower_r_live", "nested_small_dccs",
            ]].to_numpy(),
            hovertemplate=(
                "time=%{x}<br>close=%{y:.5f}<br>elapsed=%{customdata[0]:.0f}<br>"
                "progress=%{customdata[1]:.2%}<br>max progress=%{customdata[2]:.2%}<br>"
                "live lower R=%{customdata[3]:.3e}<br>"
                "nested small DCCs=%{customdata[4]:.0f}<extra></extra>"
            ),
        ), row=row_number, col=1)
        figure.add_trace(go.Scatter(
            x=[attempt.stage_start_time],
            y=[close.iloc[start_position]],
            mode="markers",
            name="Stage start",
            legendgroup="stage_start",
            showlegend=row_number == 1,
            marker={"color": "#111827", "size": 8, "symbol": "diamond"},
            hovertemplate="stage start=%{x}<extra></extra>",
        ), row=row_number, col=1)
        figure.add_trace(go.Scatter(
            x=[attempt.stage_outcome_information_time],
            y=[close.iloc[resolution_position]],
            mode="markers",
            name="Resolution",
            legendgroup="resolution",
            showlegend=row_number == 1,
            marker={"color": "#7c3aed", "size": 8, "symbol": "x"},
            hovertemplate="outcome becomes known=%{x}<extra></extra>",
        ), row=row_number, col=1)
        figure.update_yaxes(title_text="Close", row=row_number, col=1)

    figure.update_layout(
        title=f"{SYMBOL}: causal live-stage casebook",
        template="plotly_white",
        height=max(650, 330 * len(attempts)),
        hovermode="closest",
        legend={"orientation": "h", "y": 1.04, "x": 0.0},
    )
    return figure


plot_live_stage_casebook(prices, live_panel, casebook_attempts).show()

# Causal live-development completion gate

Before moving to four thresholds or a probabilistic classifier, be able to explain:

1. Why the initiating small event ID remains the lineage ID through both stages.
2. Why the medium event ID is a hand-off key rather than a replacement seed.
3. Why later small DCCs during Stage 2 are nested activity rather than new roots.
4. Why target progress can already be positive at checkpoint zero.
5. Why every live trajectory stops before its resolution bar.
6. Which feature timestamp makes each measurement causal.
7. Which DCC makes the Stage-1 or Stage-2 outcome knowable.
8. Why Stage-1 propagation can be known while terminal depth remains unknown.
9. Why later checkpoints contain survival-selected cohorts.
10. Why outcome-conditioned medians and AUCs are diagnostic hindsight rather than deployable forecasts.
11. Whether live progress adds meaningful separation beyond elapsed time and completed initiation geometry (Live Audit 2b).
12. Whether any apparent relationship survives direction, period, symbol, and threshold changes.
13. Why the active lower and upper clocks cannot fire inside a pre-resolution panel, so only activity on clocks below the lower clock is informative.

The next decision should be evidence-led: either reproduce this causal live structure on another market or threshold ladder, or extend the hierarchy only after these state and timing distinctions are intuitive.